# <center>🤖 Modèle de Prédiction du Défaut de Crédit</center>
<center><i>Pipeline scikit-learn — Classification binaire</i></center>

---

Ce notebook construit un **pipeline de machine learning complet** pour prédire le défaut de paiement.

**Plan :**
1. Chargement et nettoyage (reprise de l'EDA)
2. Feature engineering
3. Construction du Pipeline sklearn
4. Entraînement et comparaison de modèles
5. Évaluation détaillée du meilleur modèle
6. Optimisation des hyperparamètres (GridSearchCV)
7. Analyse de l'importance des features

## 1. Imports et chargement des données

In [1]:
# ── Librairies standard ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Scikit-learn : pipeline & préprocessing ─────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

# ── Scikit-learn : modèles ──────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# ── Scikit-learn : évaluation ───────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)

print("✅ Imports OK")

ModuleNotFoundError: No module named 'pandas'

In [2]:
# ── Chargement du CSV ───────────────────────────────────────────────────────
df = pd.read_csv("credit_card_default.csv")
df = df.drop(columns=['predicted_default_payment_next_month'], errors='ignore')

print(f"Lignes : {df.shape[0]} | Colonnes : {df.shape[1]}")
df.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'credit_card_default.csv'

## 2. Nettoyage (reprise de l'EDA)

In [ ]:
# ── Conversion des colonnes de statut de paiement en float ──────────────────
pay_cols = ['pay_0', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6']
df[pay_cols] = df[pay_cols].astype(float)

# ── Suppression des modalités non documentées (voir EDA) ───────────────────
df = df[~df['education_level'].isin([0, 4, 5, 6])]
df = df[~df['marital_status'].isin([0, 3])]

print(f"Lignes après nettoyage : {df.shape[0]}")
print(f"Taux de défaut : {df['default_payment_next_month'].mean()*100:.1f}%")

## 3. Feature Engineering

On crée des variables synthétiques pour enrichir le signal :
- **`max_delay`** : retard maximum observé sur les 6 mois (signal fort d'après l'EDA)
- **`mean_delay`** : retard moyen sur 6 mois
- **`total_bill`** : montant total facturé sur 6 mois
- **`total_paid`** : montant total remboursé sur 6 mois
- **`repay_ratio`** : ratio remboursement / facturation (capacité de remboursement)
- **`util_rate`** : taux d'utilisation du crédit (bill_amt_1 / limit_bal)

In [ ]:
bill_cols  = ['bill_amt_1','bill_amt_2','bill_amt_3','bill_amt_4','bill_amt_5','bill_amt_6']
amt_cols   = ['pay_amt_1','pay_amt_2','pay_amt_3','pay_amt_4','pay_amt_5','pay_amt_6']
delay_cols = ['pay_0','pay_2','pay_3','pay_4','pay_5','pay_6']

df['max_delay']    = df[delay_cols].max(axis=1)
df['mean_delay']   = df[delay_cols].mean(axis=1)
df['total_bill']   = df[bill_cols].sum(axis=1)
df['total_paid']   = df[amt_cols].sum(axis=1)
df['repay_ratio']  = df['total_paid'] / (df['total_bill'].replace(0, np.nan))
df['repay_ratio']  = df['repay_ratio'].fillna(0).clip(0, 5)  # plafonner les ratios aberrants
df['util_rate']    = df['bill_amt_1'] / df['limit_bal'].replace(0, np.nan)
df['util_rate']    = df['util_rate'].fillna(0).clip(0, 5)

print("✅ Features créées :", ['max_delay','mean_delay','total_bill','total_paid','repay_ratio','util_rate'])

## 4. Définition des features et séparation Train / Test

In [ ]:
TARGET = 'default_payment_next_month'

# ── Features numériques ─────────────────────────────────────────────────────
NUM_FEATURES = (
    ['limit_bal', 'age']
    + bill_cols
    + amt_cols
    + delay_cols
    + ['max_delay','mean_delay','total_bill','total_paid','repay_ratio','util_rate']
)

# ── Features catégorielles (encodage OHE) ───────────────────────────────────
CAT_FEATURES = ['sex', 'education_level', 'marital_status']

ALL_FEATURES = NUM_FEATURES + CAT_FEATURES

X = df[ALL_FEATURES]
y = df[TARGET]

# ── Split stratifié pour conserver le ratio de défaut ───────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")
print(f"Taux de défaut — Train : {y_train.mean()*100:.1f}% | Test : {y_test.mean()*100:.1f}%")

## 5. Construction du Pipeline sklearn

Le pipeline encapsule **toutes les transformations** et le modèle dans un seul objet,
ce qui garantit qu'aucune fuite de données entre train et test n'est possible.

In [ ]:
# ── Transformateur numérique : imputation + normalisation ───────────────────
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # robuste aux outliers
    ('scaler',  StandardScaler()),                   # nécessaire pour LR et SVM
])

# ── Transformateur catégoriel : imputation + encodage OHE ──────────────────
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# ── ColumnTransformer : applique chaque transformateur aux bonnes colonnes ──
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, NUM_FEATURES),
    ('cat', cat_transformer, CAT_FEATURES),
])

print("✅ Préprocesseur défini")
print(f"   Colonnes numériques : {len(NUM_FEATURES)}")
print(f"   Colonnes catégorielles : {len(CAT_FEATURES)}")

## 6. Comparaison de plusieurs modèles

On compare 3 modèles via **validation croisée 5 folds** sur le score ROC-AUC
(plus adapté qu'accuracy face au déséquilibre de classes 79%/21%).

In [ ]:
# ── Dictionnaire des modèles candidats ─────────────────────────────────────
models = {
    'Régression Logistique': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest':         RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}

for name, model in models.items():
    # Assemblage du pipeline complet : préprocesseur + modèle
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier',   model),
    ])
    # Validation croisée 5 folds — métrique ROC-AUC
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    results[name] = scores
    print(f"{name:30s} → ROC-AUC = {scores.mean():.4f} ± {scores.std():.4f}")

print("\n✅ Comparaison terminée")

In [ ]:
# ── Visualisation des scores de validation croisée ─────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

means = [v.mean() for v in results.values()]
stds  = [v.std()  for v in results.values()]
names = list(results.keys())

bars = ax.barh(names, means, xerr=stds, color=['#4C72B0','#55A868','#DD8452'],
               edgecolor='white', capsize=5)

for bar, m in zip(bars, means):
    ax.text(m + 0.002, bar.get_y() + bar.get_height()/2,
            f"{m:.4f}", va='center', fontweight='bold')

ax.set_xlabel('ROC-AUC (CV 5 folds)')
ax.set_title('Comparaison des modèles — Validation croisée', fontweight='bold')
ax.set_xlim(0.5, 1.0)
plt.tight_layout()
plt.show()

## 7. Évaluation du meilleur modèle sur le Test Set

On retient le **Gradient Boosting** (généralement le plus performant sur ce type de données).
Adaptez en fonction des résultats de la cellule précédente.

In [ ]:
# ── Construction et entraînement du pipeline final ──────────────────────────
best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   GradientBoostingClassifier(n_estimators=100, random_state=42)),
])

best_pipeline.fit(X_train, y_train)

# ── Prédictions ─────────────────────────────────────────────────────────────
y_pred      = best_pipeline.predict(X_test)
y_pred_proba = best_pipeline.predict_proba(X_test)[:, 1]

print("=== Rapport de classification ===")
print(classification_report(y_test, y_pred, target_names=['Pas de défaut', 'Défaut']))
print(f"ROC-AUC : {roc_auc_score(y_test, y_pred_proba):.4f}")

In [ ]:
# ── Matrice de confusion + Courbe ROC ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice de confusion
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Pas de défaut', 'Défaut'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Matrice de confusion', fontweight='bold')

# Courbe ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='#4C72B0', lw=2, label=f'ROC-AUC = {auc:.4f}')
axes[1].plot([0,1],[0,1],'--', color='grey', label='Aléatoire')
axes[1].set_xlabel('Taux de faux positifs (FPR)')
axes[1].set_ylabel('Taux de vrais positifs (TPR)')
axes[1].set_title('Courbe ROC', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Optimisation des hyperparamètres (GridSearchCV)

In [ ]:
# ── Grille de recherche pour Gradient Boosting ──────────────────────────────
# Les paramètres du classifier dans le pipeline se préfixent avec 'classifier__'
param_grid = {
    'classifier__n_estimators':   [100, 200],
    'classifier__max_depth':      [3, 5],
    'classifier__learning_rate':  [0.05, 0.1],
    'classifier__subsample':      [0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator  = best_pipeline,
    param_grid = param_grid,
    cv         = 3,             # 3 folds pour la rapidité (augmenter à 5 si le temps le permet)
    scoring    = 'roc_auc',
    n_jobs     = -1,
    verbose    = 1,
)

grid_search.fit(X_train, y_train)

print(f"\n✅ Meilleurs paramètres : {grid_search.best_params_}")
print(f"   Meilleur ROC-AUC (CV) : {grid_search.best_score_:.4f}")

In [ ]:
# ── Évaluation finale du modèle optimisé ────────────────────────────────────
best_model = grid_search.best_estimator_
y_pred_opt       = best_model.predict(X_test)
y_pred_proba_opt = best_model.predict_proba(X_test)[:, 1]

print("=== Rapport de classification — Modèle optimisé ===")
print(classification_report(y_test, y_pred_opt, target_names=['Pas de défaut', 'Défaut']))
print(f"ROC-AUC : {roc_auc_score(y_test, y_pred_proba_opt):.4f}")

## 9. Importance des features

In [ ]:
# ── Récupération des noms de features après transformation OHE ──────────────
ohe_feature_names = (
    best_model.named_steps['preprocessor']
    .named_transformers_['cat']
    .named_steps['ohe']
    .get_feature_names_out(CAT_FEATURES)
    .tolist()
)
all_feature_names = NUM_FEATURES + ohe_feature_names

# ── Importances du modèle final ─────────────────────────────────────────────
importances = best_model.named_steps['classifier'].feature_importances_

feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False)

# ── Visualisation : Top 20 features ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
feat_imp.head(20).plot(kind='barh', ax=ax, color='#4C72B0', edgecolor='white')
ax.invert_yaxis()
ax.set_title('Top 20 — Importance des features (Gradient Boosting)', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 10 features :")
print(feat_imp.head(10).to_string())

## 10. Sauvegarder le pipeline

On sauvegarde le pipeline final pour une utilisation en production.

In [ ]:
import joblib

# Sauvegarde du pipeline complet (préprocesseur + modèle)
joblib.dump(best_model, 'credit_default_pipeline.pkl')
print("✅ Pipeline sauvegardé : credit_default_pipeline.pkl")

# Chargement et vérification
loaded_pipeline = joblib.load('credit_default_pipeline.pkl')
test_pred = loaded_pipeline.predict_proba(X_test.head(5))[:, 1]
print(f"Vérification — 5 premières probabilités de défaut : {test_pred.round(3)}")

---

## 🎯 Récapitulatif

| Étape | Choix technique | Justification |
|---|---|---|
| **Préprocessing** | `StandardScaler` + `OneHotEncoder` dans un `ColumnTransformer` | Zéro fuite train/test, tout dans le pipeline |
| **Feature engineering** | `max_delay`, `repay_ratio`, `util_rate` | Insights de l'EDA : retard = signal fort |
| **Modèle** | Gradient Boosting | Robuste aux outliers, capture les non-linéarités |
| **Évaluation** | ROC-AUC + CV 5 folds | Adapté au déséquilibre 79/21 |
| **Optimisation** | `GridSearchCV` | Recherche systématique des meilleurs hyperparamètres |
| **Déploiement** | `joblib` | Pipeline entier sérialisé, prêt pour la production |